# CSE 151B Competition

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # run in only first time

# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !~/.local/bin/uv venv .venv --seed

# # activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

# # Install dependencies — this is fast thanks to uv's parallel resolver
# # adding more constraints from piazza
# !~/.local/bin/uv pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# !~/.local/bin/uv pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter accelerate -c constraints.txt


# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:

!{sys.executable} -m pip install sympy numpy "transformers>=4.51.0" vllm tqdm bitsandbytes accelerate "antlr4-python3-runtime==4.11.1"

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

import vllm
print("vllm:", vllm.__version__)

import transformers
print("transformers:", transformers.__version__)

import bitsandbytes
print("bitsandbytes:", bitsandbytes.__version__)

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)

In [ ]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"
DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/results.jsonl"
MAX_TOKENS  = 4096

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## I added for cuda verification
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")
##


import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
try:
    from vllm import LLM, SamplingParams
except ModuleNotFoundError:
    LLM = None
    SamplingParams = None
    print("vLLM is not installed; using Transformers backend instead.")
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Solve the problem step-by-step, showing all necessary work. "
    "Double-check your calculations before finalizing. "
    "Do not repeat a step you have already completed. "
    "Simplify all fractions and radicals. Do not use decimals unless the problem requires it. "
    "If the problem has no solution, write \\boxed{None}. "
    "If the problem is ambiguous, state your assumption in one sentence, then solve. "
    "Once you are confident in your final answer, write it inside \\boxed{}. "
    "Your final answer must appear exactly once inside \\boxed{}. "
    "Do not place intermediate results inside \\boxed{}. "
    "For multiple sub-answers, use a single \\boxed{} with comma separation, e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician solving a multiple-choice problem. "
    "Work efficiently: identify the correct answer with minimal necessary steps. "
    "Do not evaluate every option exhaustively — solve the problem directly, then match to an option. "
    "You MUST choose exactly one letter. If unsure, pick the most likely option. "
    "End your response with the answer on its own line in the exact format \\boxed{X}, "
    "where X is a single letter. The \\boxed{} must be the very last thing you write."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
# !~/151B_SP26_Competition/.venv/bin/pip install "transformers>=4.51.0" --upgrade
# !~/151B_SP26_Competition/.venv/bin/pip install "vllm>=0.8.0" --upgrade

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen3-4B-Thinking-2507",
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.90, # bump from 0.50 using more VRAM
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=64, # parallel sequences
    )

sampling_params = SamplingParams(
    max_tokens=4096, # reduce from 32768 unless we need full-length thinking, it COULD increase accuracy, but it would also take a while longer to generate.
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    n=3, # generate K self-consistency samples in ONE call
)

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
from collections import Counter

prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate (n=3 samples per question, from sampling_params)
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

# Keep ALL samples per question (list of lists), instead of just the first
all_samples = [[o.text.strip() for o in out.outputs] for out in outputs]

# Backwards-compatible: responses = first sample of each
responses = [samples[0] for samples in all_samples]

# Preview
for i in range(min(3, len(all_samples))):
    print(f"\n── Question {i} (id={data[i].get('id')}) — {len(all_samples[i])} samples ──")
    print(all_samples[i][0][:300], "...")

In [ ]:
responses[0] #Just to check and quickly gauge what a full answer looks like

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
from collections import Counter

def extract_letter_strict(text: str) -> str:
    """Extract a boxed letter. Returns '' if no clean boxed letter found (NO garbage fallback)."""
    m = re.search(r"\\boxed\{\s*([A-Za-z])\s*\}", text)
    if m:
        return m.group(1).upper()
    return ""  # no fallback — empty means "this sample didn't vote"

def extract_letter_loose(text: str) -> str:
    """Looser extraction for the FINAL fallback only (used when no sample boxed cleanly)."""
    m = re.search(r"\\boxed\{\s*([A-Za-z])\s*\}", text)
    if m:
        return m.group(1).upper()
    # last-resort: 'answer is X' / 'the answer: X'
    m2 = re.search(r"answer\s*(?:is|:)?\s*\(?([A-Za-z])\)?", text, re.IGNORECASE)
    if m2:
        return m2.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text)
    return matches[-1] if matches else ""

def extract_boxed(text: str) -> str:
    """Extract content of the last \\boxed{...}, handling nested braces."""
    idx = text.rfind(r"\boxed{")
    if idx == -1:
        return ""
    i = idx + len(r"\boxed{")
    depth = 1
    out = []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(c)
        i += 1
    return "".join(out).strip()

def majority_vote(samples, is_mcq):
    """MCQ: vote over clean boxed letters. Free-form: use first sample (let Judger parse)."""
    if is_mcq:
        votes = [extract_letter_strict(s) for s in samples]
        votes = [v for v in votes if v]
        if votes:
            winner, _ = Counter(votes).most_common(1)[0]
            for s in samples:
                if extract_letter_strict(s) == winner:
                    return s
        return samples[0]
    else:
        # Free-form: voting on raw strings is unreliable (1/2 vs 0.5 etc).
        # Just return the first sample; the Judger handles extraction + equivalence.
        return samples[0]

    # No sample produced a clean boxed answer → fall back to first sample,
    # and let the loose extractor / judger try on it.
    return samples[0]

def score_mcq(response: str, gold_letter: str) -> bool:
    # use loose here so a single-sample fallback still gets a fair shot
    return extract_letter_loose(response) == gold_letter.strip().upper()

# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, samples in tqdm(zip(data, all_samples), total=len(all_samples), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    voted_response = majority_vote(samples, is_mcq)

    if is_mcq:
        correct = score_mcq(voted_response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=voted_response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": voted_response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = False # Set to False when running on the private test set, True for testing on public.

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## 10. `run_inference()` — Single Entry Point

Wraps the full pipeline end-to-end: loads model, runs inference on the private dataset, applies all post-processing (majority voting, answer extraction), and outputs the final submission CSV.

All hyperparameters below match the final submission values.

In [1]:
import json
import os
import re
import sys
import csv
from pathlib import Path
from collections import Counter
from typing import Optional


def run_inference(
    data_path: str = "data/private.jsonl",
    output_csv: str = "results/submission.csv",
):
    """
    Full inference pipeline — single entry point.
    Loads model, runs inference on data_path, applies majority-vote post-processing,
    and writes the final submission CSV to output_csv.
    """

    # Final hyperparameters (match submission) 
    MODEL_ID              = "Qwen/Qwen3-4B-Thinking-2507"
    GPU_ID                = "0"
    MAX_TOKENS            = 4096
    TEMPERATURE           = 0.6
    TOP_P                 = 0.95
    TOP_K                 = 20
    N_SAMPLES             = 3      # self-consistency samples per question
    GPU_MEMORY_UTIL       = 0.90
    MAX_MODEL_LEN         = 16384
    MAX_NUM_SEQS          = 64

    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

    # Prompts 
    SYSTEM_PROMPT_MATH = (
        "You are an expert mathematician. "
        "Solve the problem step-by-step, showing all necessary work. "
        "Double-check your calculations before finalizing. "
        "Do not repeat a step you have already completed. "
        "Simplify all fractions and radicals. Do not use decimals unless the problem requires it. "
        "If the problem has no solution, write \\boxed{None}. "
        "If the problem is ambiguous, state your assumption in one sentence, then solve. "
        "Once you are confident in your final answer, write it inside \\boxed{}. "
        "Your final answer must appear exactly once inside \\boxed{}. "
        "Do not place intermediate results inside \\boxed{}. "
        "For multiple sub-answers, use a single \\boxed{} with comma separation, e.g. \\boxed{3, 7}."
    )
    SYSTEM_PROMPT_MCQ = (
        "You are an expert mathematician solving a multiple-choice problem. "
        "Work efficiently: identify the correct answer with minimal necessary steps. "
        "Do not evaluate every option exhaustively — solve the problem directly, then match to an option. "
        "You MUST choose exactly one letter. If unsure, pick the most likely option. "
        "End your response with the answer on its own line in the exact format \\boxed{X}, "
        "where X is a single letter. The \\boxed{} must be the very last thing you write."
    )

    def build_prompt(question: str, options: Optional[list]):
        if options:
            labels    = [chr(65 + i) for i in range(len(options))]
            opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
            return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
        return SYSTEM_PROMPT_MATH, question

    # Answer extraction helpers 
    def extract_letter_strict(text: str) -> str:
        m = re.search(r"\\boxed\{\s*([A-Za-z])\s*\}", text)
        return m.group(1).upper() if m else ""

    def extract_letter_loose(text: str) -> str:
        m = re.search(r"\\boxed\{\s*([A-Za-z])\s*\}", text)
        if m:
            return m.group(1).upper()
        m2 = re.search(r"answer\s*(?:is|:)?\s*\(?([A-Za-z])\)?", text, re.IGNORECASE)
        if m2:
            return m2.group(1).upper()
        matches = re.findall(r"\b([A-Z])\b", text)
        return matches[-1] if matches else ""

    def majority_vote(samples, is_mcq):
        if is_mcq:
            votes = [v for v in (extract_letter_strict(s) for s in samples) if v]
            if votes:
                winner, _ = Counter(votes).most_common(1)[0]
                for s in samples:
                    if extract_letter_strict(s) == winner:
                        return s
        return samples[0]

    # Load model 
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token

    llm = LLM(
        model=MODEL_ID,
        quantization="bitsandbytes",
        load_format="bitsandbytes",
        enable_prefix_caching=False,
        gpu_memory_utilization=GPU_MEMORY_UTIL,
        max_model_len=MAX_MODEL_LEN,
        trust_remote_code=True,
        max_num_seqs=MAX_NUM_SEQS,
    )
    sampling_params = SamplingParams(
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        n=N_SAMPLES,
    )

    # Load data
    data = [json.loads(line) for line in open(data_path)]
    print(f"Loaded {len(data)} questions.")

    # Build prompts
    prompts = []
    for item in data:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    # Run inference
    print(f"Running inference on {len(prompts)} questions...")
    outputs = llm.generate(prompts, sampling_params=sampling_params)
    all_samples = [[o.text.strip() for o in out.outputs] for out in outputs]

    # Post-process & build records
    records = []
    for item, samples in zip(data, all_samples):
        is_mcq = bool(item.get("options"))
        response = majority_vote(samples, is_mcq)
        records.append({
            "id":       item.get("id"),
            "is_mcq":   is_mcq,
            "response": response,
        })

    # Write submission CSV 
    out_path = Path(output_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "response"])
        writer.writeheader()
        for r in records:
            writer.writerow({"id": r["id"], "response": r["response"]})

    print(f"Submission saved to {out_path} ({len(records)} records).")
    return records



In [2]:
run_inference(data_path="data/private.jsonl", output_csv="results/results.csv")

INFO 06-01 07:58:55 [utils.py:278] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 64, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 06-01 07:58:56 [model.py:617] Resolved architecture: Qwen3ForCausalLM
INFO 06-01 07:58:56 [model.py:1752] Using max model len 16384
INFO 06-01 07:58:56 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-01 07:58:58 [vllm.py:977] Asynchronous scheduling is enabled.
INFO 06-01 07:58:58 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=27517) INFO 06-01 07:59:03 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', sk

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=27517) /opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=27517)   torch._check_is_size(blocksize)


(EngineCore pid=27517) INFO 06-01 07:59:44 [model_runner.py:295] Model loading took 2.71 GiB and 40.699901 seconds
(EngineCore pid=27517) INFO 06-01 07:59:52 [backends.py:1089] Using cache directory: /home/jupyter/.cache/vllm/torch_compile_cache/702fca2bcf/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=27517) INFO 06-01 07:59:52 [backends.py:1148] Dynamo bytecode transform time: 7.13 s
(EngineCore pid=27517) INFO 06-01 07:59:56 [backends.py:378] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=27517) INFO 06-01 08:00:03 [backends.py:393] Compiling a graph for compile range (1, 8192) takes 10.10 s
(EngineCore pid=27517) INFO 06-01 08:00:03 [decorators.py:311] Directly load AOT compilation from path /home/jupyter/.cache/vllm/torch_compile_cache/torch_aot_compile/073873b2a88e71858306ff49cac5af049bc58302ed6b1853139daa9e29ec0d04/rank_0_0/model
(EngineCore pid=27517) INFO 06-01 08:00:03 [monitor.py:53] torch.compile took 18.17 s in total
(EngineCore pid=27

Capturing CUDA graphs (PIECEWISE):  95%|█████████▍| 18/19 [00:04<00:00,  4.34it/s]/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=27517)   torch._check_is_size(blocksize)
Capturing CUDA graphs (FULL): 100%|██████████| 11/11 [00:02<00:00,  4.52it/s]


...

Rendering prompts:   0%|          | 0/943 [00:00<?, ?it/s]

(EngineCore pid=27517) WARNING 06-01 08:00:15 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts:   0%|          | 0/2829 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=27517) INFO 06-01 08:00:37 [core.py:1266] Shutdown initiated (timeout=0)
(EngineCore pid=27517) INFO 06-01 08:00:37 [core.py:1271] Aborting 2829 requests
(EngineCore pid=27517) INFO 06-01 08:00:37 [core.py:1289] Shutdown complete


KeyboardInterrupt: 